# Natural Twi Voice: F5-TTS Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theonlyamos/ghana-languages-speech-to-speech-pipeline/blob/claude/project-thread-htkyqr/train_tts_f5.ipynb)

Fine-tunes [F5-TTS](https://github.com/SWivid/F5-TTS) (v1 Base) into a natural-sounding Twi (or Ewe) voice.
Runs on Google Colab or any machine with a ~24 GB NVIDIA GPU (L4, A100, RTX 3090/4090).

**On Colab:** Runtime → Change runtime type → **A100 GPU**, then Runtime → Run all. A full run takes about 8–10 hours.

Each step calls `scripts/train_tts_f5.py`, so the notebook and the command line always do the same thing.

| Step | What happens |
|---|---|
| 1 | Check the GPU, get the code, install dependencies |
| 2 | Choose the voice, language and data sources |
| 3 | Download BibleTTS and prepare all sources into one F5 dataset |
| 4 | Fine-tune |
| 5 | Listen to the result |
| 6 | Save the voice |

**Licensing:** the pretrained F5 weights are CC-BY-NC, and so are some data sources (see step 2), so the trained voice is for non-commercial use.

## 1. Setup

In [ ]:
import os, subprocess, sys, time
from IPython.display import Pretty, display

# Fewer progress redraws at the source too (download, prepare, training).
os.environ["TQDM_MININTERVAL"] = "5"

def sh(cmd, every=5.0):
    """Run a shell command, streaming its output into the notebook; stop if it fails.

    Progress bars (redrawn with carriage returns) are shown as ONE line updated in
    place at most every `every` seconds: printing every redraw freezes the browser tab.
    """
    print(f"$ {cmd}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    buf, bar, latest, last = "", None, "", 0.0
    while chunk := p.stdout.read1(65536):
        buf = (buf + chunk.decode("utf-8", "replace")).replace("\r\n", "\n")
        while True:
            cuts = [j for j in (buf.find("\n"), buf.find("\r")) if j >= 0]
            if not cuts:
                break
            i = min(cuts)
            if buf[i] == "\r" and i == len(buf) - 1:
                break  # may be the first half of "\r\n"; wait for more output
            seg, sep, buf = buf[:i], buf[i], buf[i + 1:]
            if not seg.strip():
                continue
            if sep == "\r":  # a progress redraw
                latest = seg
                if bar is None:
                    bar, last = display(Pretty(seg), display_id=True), time.time()
                elif time.time() - last >= every:
                    bar.update(Pretty(seg))
                    last = time.time()
            elif bar is not None:  # the bar's final state
                bar.update(Pretty(seg))
                bar = None
            else:
                print(seg, flush=True)
    if bar is not None:
        bar.update(Pretty(latest))
    if buf.strip():
        print(buf, flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")


In [ ]:
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
REPO_URL = "https://github.com/theonlyamos/ghana-languages-speech-to-speech-pipeline"
BRANCH = "claude/project-thread-htkyqr"  # switch to "main" once this work is merged

if IN_COLAB:
    if not Path("/content/ghana/.git").exists():
        sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/ghana")
    os.chdir("/content/ghana")
elif not Path("scripts/train_tts_f5.py").exists():
    raise SystemExit("Open this notebook from the repository root.")

sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
# F5-TTS + the project's other dependencies (torch/CUDA are preinstalled on Colab).
sh(f"{sys.executable} -m pip install -q f5-tts datasets soundfile pandas openpyxl requests tqdm")

**Google Drive (recommended).** Colab VMs are wiped when the session ends. With Drive on, training
checkpoints are kept there (needs ~7 GB free), so a disconnected run resumes from `model_last.pt`.


In [ ]:
USE_DRIVE = True  # recommended for long runs: survives disconnects

# Only checkpoints go to Drive (~7 GB). The prepared audio (~30 GB of small files) stays on
# the VM's fast local disk: it would overflow a free 15 GB Drive and slow training badly.
# After a disconnect, re-run prepare (about an hour), and training resumes from Drive.
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    keep = Path("/content/drive/MyDrive/ghana_f5")
    stale = Path("data/processed/tts")  # older notebook versions linked this to Drive too
    if stale.is_symlink():
        stale.unlink()
    sub = "models/tts_f5"
    (keep / sub).mkdir(parents=True, exist_ok=True)
    local = Path(sub)
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.is_symlink():
        if local.exists():
            sh(f"cp -rn {local}/. {keep / sub}/ && rm -rf {local}")
        local.symlink_to(keep / sub)
    print("Checkpoints are stored in", keep / sub)


## 2. Choose the voice and data

| Source | Data | License |
|---|---|---|
| BibleTTS (`BIBLE_LANG`) | Studio Bible readings, one speaker | CC-BY-SA-4.0 |
| `waxal_twi` / `waxal_fante` / `waxal_ewe` | Google WAXAL studio voice actors | CC-BY-4.0 |
| `loud_lounge` | ~65 h conversational podcast Twi/English | CC-BY-NC-4.0 |
| `twi_multispeaker` | ~90 h single-speaker clips, quality-filtered | CC-BY-NC-4.0 |

The **reference clip** that sets the output voice comes from the first source (BibleTTS). The other
sources teach the model natural Twi rhythm. Set `LIMIT = 200` for a quick smoke run (200 clips per source).

In [ ]:
NAME = "twi"                 # voice name -> models/tts_f5/<NAME>
BIBLE_LANG = "asante_twi"    # asante_twi | akuapem_twi | ewe
SOURCES = ["waxal_twi", "loud_lounge", "twi_multispeaker"]   # for Ewe: ["waxal_ewe"]
COMMERCIAL_ONLY = False      # True drops the CC-BY-NC sources
LIMIT = 0                    # clips per source, 0 = all
EPOCHS = 20
BATCH_FRAMES = 4000          # lower to 2000-3000 if you hit CUDA out-of-memory

## 3. Download and prepare the data

In [ ]:
from utils.data_processing import DatasetDownloader

bible_dir = Path("data/raw/bibletis") / BIBLE_LANG
if not any(bible_dir.rglob("*.txt")):
    DatasetDownloader().download_bibletis([BIBLE_LANG])
print("BibleTTS:", bible_dir, "->", sum(1 for _ in bible_dir.rglob("*.txt")), "transcripts")

In [ ]:
args = " ".join(f"--source {s}" for s in SOURCES)
if COMMERCIAL_ONLY:
    args += " --commercial-only"
# Hugging Face sources are streamed; only the kept clips are written to disk as 24 kHz wavs.
sh(f"{sys.executable} scripts/train_tts_f5.py prepare --name {NAME} --input {bible_dir} {args} --limit {LIMIT}")

In [ ]:
import json
from IPython.display import Audio, display

data_dir = Path("data/processed/tts") / f"f5_{NAME}"
for s in json.loads((data_dir / "sources.json").read_text(encoding="utf-8")):
    print(f"{s['source']:<40} {s['clips']:>7} clips {s['hours']:>7} h   {s['license']}")
print("\nReference clip (the voice the model will speak in):")
print((data_dir / "ref.txt").read_text(encoding="utf-8"))
display(Audio(str(data_dir / "ref.wav")))

## 4. Fine-tune

Saves `model_last.pt` regularly, so if Colab disconnects you can re-run this cell and it continues
where it stopped (keep the files with Drive, step 1). Expect several hours for the full data on an L4 or A100.

In [ ]:
# On Drive keep only model_last.pt (~5 GB); numbered checkpoints would overflow a 15 GB Drive.
keep_n = 0 if USE_DRIVE else 2
sh(f"{sys.executable} scripts/train_tts_f5.py train --name {NAME} --epochs {EPOCHS} --batch-frames {BATCH_FRAMES} --keep-checkpoints {keep_n}")


## 5. Listen

In [ ]:
TEXT = "Maakye, wo ho te s\u025bn? Me din de Kofi."  # "Good morning, how are you? My name is Kofi."

out = Path("outputs") / f"{NAME}_sample.wav"
out.parent.mkdir(exist_ok=True)
sh(f'{sys.executable} scripts/infer.py --task speak --tts-model-path models/tts_f5/{NAME} --text "{TEXT}" --output {out}')
display(Audio(str(out)))

## 6. Save the voice

Inference needs only `model_last.pt`, `vocab.txt`, `ref.wav` and `ref.txt` (`sources.json` records the training data).

In [ ]:
voice = Path("models/tts_f5") / NAME
files = [voice / f for f in ("model_last.pt", "vocab.txt", "ref.wav", "ref.txt", "sources.json") if (voice / f).exists()]
archive = Path(f"{NAME}_f5_voice.tar")
sh(f"tar -cf {archive} " + " ".join(str(f) for f in files))
print(f"{archive}: {archive.stat().st_size / 1e9:.1f} GB")

if IN_COLAB and not USE_DRIVE:
    from google.colab import files as colab_files
    colab_files.download(str(archive))